In [1]:
# Import libraries
from pathlib import Path

import pandas as pd 
import numpy as np
import xarray as xr

In [2]:
"""
    - **Download:** https://www.thermogis.nl/sites/default/files/2026-05/for_external_use.zip)
    - Navigate to: `ThermoGIS_grids_2_5_1 > 6_Permian`
"""
# Input and output paths
ROOT      = Path().resolve().parent
RAW       = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed' 
PROCESSED.mkdir(parents=True, exist_ok=True)

In [3]:
# Directory Paths
DATA_DIR = Path(r'C:\Users\sadar\Downloads\thermogis_data') 
GRID_DIR = DATA_DIR / 'ThermoGIS_grids_2_5_1' 
PERMIAN_GRID_DIR = GRID_DIR / '6_Permian'

# Utrecht Province Clip 
X_MIN, X_MAX = 100_000, 180_000
Y_MIN, Y_MAX = 420_000, 490_000

In [4]:
# Find Permian Files
all_permian = list(PERMIAN_GRID_DIR.rglob('*.nc'))
print(f'Total Permian .nc files: {len(all_permian)}')

# Formations in Permian folder
formations = sorted(set(f.relative_to(PERMIAN_GRID_DIR).parts[0] for f in all_permian))
print(f'\nFormations in 6_Permian folder:')
for fm in formations:
    print(f'  {fm}')

Total Permian .nc files: 132

Formations in 6_Permian folder:
  Lower Slochteren Mb (ROSLL)
  Slochteren Fm & Upper Slochteren Mb (ROSL&ROSLU)
  Upper Rotliegend Gp (RO)


In [5]:
# Slochteren Fm & Upper Slochteren Mb (ROSL&ROSLU) and Lower Slochteren Mb (ROSLL) are members of the Upper Rotliegend Gp (RO)
formations = [f for f in formations if f != 'Upper Rotliegend Gp (RO)']
for fm in formations:
    print(f'  {fm}')

  Lower Slochteren Mb (ROSLL)
  Slochteren Fm & Upper Slochteren Mb (ROSL&ROSLU)


In [6]:
# Formations with Utrecht coverage 

utrecht_formations = []

for formation in formations:
    sample_files = [
        f for f in all_permian
        if f.parts[-3] == formation
        and f.parent.name == 'BaseCase'
    ]
    if not sample_files:
        print(f'  {formation:55s} - no BaseCase files')
        continue

    f = sample_files[0] 
    ds = xr.open_dataset(f) 
    coords = list(ds.coords) 
    x_coord = next((c for c in coords if c.lower() in ('x', 'xrdx', 'easting')), None)
    y_coord = next((c for c in coords if c.lower() in ('y', 'rdy', 'northing')), None)


    if not x_coord or not y_coord:
        print(f'  {formation:55s} - unrecognized coords:  {coords}')
        continue

    x_vals = ds[x_coord].values 
    y_vals = ds[y_coord].values 

    # Check overlap with Utrecht
    x_overlap = (x_vals.max() >= X_MIN) and (x_vals.min() <= X_MAX)
    y_overlap = (y_vals.max() >= Y_MIN) and (y_vals.min() <= Y_MAX ) 

    if x_overlap and y_overlap:
        #Check non-nulls
        ds_ut = ds.sel({
            x_coord: (x_vals >= X_MIN) & (x_vals <= X_MAX),
            y_coord: (y_vals >= Y_MIN) & (y_vals <= Y_MAX)
        })
        data = ds_ut['data'].values.flatten()
        n_valid = int(np.sum(~np.isnan(data) & (data > 0)))

        status = f'✓ PRESENT - {n_valid} valid cells in Utrecht'
        utrecht_formations.append(formation) 
    else:
        status = (f'x outside Utrecht   '
                  f'(x: {x_vals.min():.0f}-{x_vals.max():.0f}, '
                  f'y: {y_vals.min():.0f}-{y_vals.max():.0f})')
    print(f'    {formation:55s} - {status}')

print(f'\n→ Formations present in Utrecht: {len(utrecht_formations)}')
for f in utrecht_formations:
    print(f'    {f}')

    Lower Slochteren Mb (ROSLL)                             - x outside Utrecht   (x: 119000-262000, y: 574000-624000)
    Slochteren Fm & Upper Slochteren Mb (ROSL&ROSLU)        - ✓ PRESENT - 4971 valid cells in Utrecht

→ Formations present in Utrecht: 1
    Slochteren Fm & Upper Slochteren Mb (ROSL&ROSLU)


In [7]:
# Present Formations Only 
# Load data
BASECASE_DIR = PERMIAN_GRID_DIR / 'Slochteren Fm & Upper Slochteren Mb (ROSL&ROSLU)' / 'BaseCase'
ROSL_ROSLU_basecase_files = [
    f for f in BASECASE_DIR.rglob('*nc')
    if f.name.upper().startswith('ROSL_ROSLU_')
]

for f in ROSL_ROSLU_basecase_files:
    print(f.name)

ROSL_ROSLU_depth.nc
ROSL_ROSLU_economic_potential.nc
ROSL_ROSLU_flow_rate_p50.nc
ROSL_ROSLU_heat_in_place.nc
ROSL_ROSLU_net_to_gross.nc
ROSL_ROSLU_permeability_p50.nc
ROSL_ROSLU_porosity.nc
ROSL_ROSLU_potential_recoverable_heat.nc
ROSL_ROSLU_power_p50.nc
ROSL_ROSLU_temperature.nc
ROSL_ROSLU_thickness_p50.nc


- Copy these .nc files to RAW path